# Laboratori 03: Ekuacionet jolineare, sistemet lineare dhe vlerat vetjake

**Lënda:** Simulim dhe Metoda Numerike  
**Kohëzgjatja:** 120 minuta  
**Mjedisi kryesor:** Python/NumPy; pjesa e dytë përmban zbatimin thelbësor në GNU Octave.

**Pyetja udhëheqëse:** Si zgjedhim një algoritëm të shpejtë pa humbur kontrollin mbi konvergjencën dhe ndjeshmërinë?

Ky laborator nuk duhet lexuar si një listë komandash. Çdo pjesë nis nga modeli dhe nga një llogaritje që mund të kryhet me dorë; kodi përdoret më pas për të zgjeruar llogaritjen, për ta verifikuar dhe për të eksploruar regjime që nuk trajtohen dot me disa hapa algjebrikë. Ritmi i punës është

$$\text{pyetje}\longrightarrow\text{model}\longrightarrow\text{derivim}\longrightarrow\text{algoritëm}\longrightarrow\text{kod}\longrightarrow\text{kontroll}.$$


## Rezultatet e të nxënit

Në fund të seancës studenti duhet të jetë në gjendje të:

- zbatojë përgjysmimin dhe Njutonin për $f(x)=0$;
- nxjerrë numrin e hapave të përgjysmimit;
- shpjegojë konvergjencën kuadratike lokale të Njutonit;
- numërojë veprimet e eliminimit të Gausit dhe të metodës fuqi;
- përdorë mbetjen dhe numrin e kushtëzimit si diagnostika;
- shpjegojë rolin e pivotimit dhe të normalizimit.


## Organizimi i seancës

| Koha | Veprimtaria | Produkti |
|---:|---|---|
| 0–15 min | Pyetja fizike dhe llogaritja e parë me dorë | Parashikim i shkruar |
| 15–40 min | Derivimi dhe algoritmi minimal | Formula dhe pseudokodi |
| 40–65 min | Zbatimi i udhëhequr në Python | Rezultat i verifikuar |
| 65–82 min | Kompleksiteti: formulë dhe matje | $T(n)$ dhe grafik kohor |
| 82–100 min | Qëndrueshmëria dhe diagnostika | Krahasim i dy rasteve |
| 100–112 min | Shembulli i plotë dhe punishte | Eksperiment i modifikuar |
| 112–120 min | GNU Octave dhe ekzekutimi lokal | Skript i riprodhueshëm |

Seksioni i shembullit të plotë është i detyrueshëm. Në punishte zgjidhen vetëm një ose dy ndryshime parametrash gjatë orës; pjesa tjetër mbetet për punë të pavarur.


## Si duhet lexuar ky laborator

Në të tria klasat e problemeve duhet të ndahen katër nocione: ekzistenca e zgjidhjes, njëtrajtshmëria e saj, konvergjenca e algoritmit dhe kriteri praktik i ndalimit. Një metodë mund të ketë rend të lartë konvergjence dhe përsëri të dështojë nga një pikë fillestare e papërshtatshme. Po kështu, një sistem mund të zgjidhet me mbetje shumë të vogël, por zgjidhja të jetë e ndjeshme sepse matrica është e keqkushtëzuar.

Gjatë leximit mos memorizoni vetëm emrat e funksioneve. Për çdo algoritëm identifikoni informacionin që kërkon, garancinë që ofron, koston e një hapi dhe diagnostikën e daljes. Përgjysmimi kërkon interval me ndryshim shenje; Njutoni kërkon derivat dhe pikë fillestare; eliminimi kërkon pivotim; metoda fuqi kërkon hendek spektral.


## 1. Tri probleme, një ide e përbashkët

Kërkimi i rrënjës, zgjidhja e sistemit linear dhe gjetja e vlerës vetjake duken si tema të ndryshme, por në të tria ndërtojmë një varg përafrimesh dhe na duhet një kriter ndalimi.

- Rrënja: $f(x_\star)=0$.
- Sistemi: $A\mathbf{x}_\star=\mathbf{b}$.
- Çifti vetjak: $A\mathbf{v}_\star=\lambda_\star\mathbf{v}_\star$.

Diagnostika më e drejtpërdrejtë është **mbetja**: $|f(x)|$, $\|A\mathbf{x}-\mathbf{b}\|$ ose $\|A\mathbf{v}-\lambda\mathbf{v}\|$. Mbetja e vogël është e nevojshme, por në një problem të keqkushtëzuar nuk garanton gjithmonë gabim të vogël në zgjidhje.


## 2. Përgjysmimi: llogaritja me dorë

Nëse $f$ është e vazhdueshme dhe $f(a)f(b)<0$, ekziston të paktën një rrënjë në $[a,b]$. Pas një hapi intervali ka gjatësi $(b-a)/2$; pas $k$ hapash,

$$L_k=\frac{b-a}{2^k}.$$

Nëse gabimi i mesit duhet të jetë jo më i madh se $\varepsilon$, mjafton

$$\frac{b-a}{2^{k+1}}\leq\varepsilon,$$

prandaj

$$k\geq\left\lceil\log_2\frac{b-a}{2\varepsilon}\right\rceil.$$

Për $f(x)=x^2-2$, $[1,2]$ dhe $\varepsilon=10^{-3}$ duhen të paktën 9 hapa. Kjo është një garanci para se të ekzekutohet kodi.


# Pjesa I — Python

## Përgatitja e mjedisit

Importet vendosen në një qelizë të vetme që eksperimenti të jetë i lexueshëm dhe i riprodhueshëm. Fara e gjeneratorit deklarohet edhe kur pjesa kryesore është deterministe; kështu çdo shtesë stokastike mund të përsëritet.


In [ ]:
import math
from time import perf_counter
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(2026)  # realizim i riprodhueshëm
np.set_printoptions(precision=6, suppress=True)


In [ ]:
def pergjysmimi(f, a, b, tol=1e-10, max_iter=100):
    fa, fb = f(a), f(b)
    if fa*fb >= 0:
        raise ValueError('Intervali nuk izolon një ndryshim shenje.')
    historia = []
    for k in range(max_iter):
        m = 0.5*(a+b)
        fm = f(m)
        historia.append((k, a, b, m, fm))
        if 0.5*(b-a) <= tol or fm == 0:
            return m, historia
        if fa*fm < 0:
            b, fb = m, fm
        else:
            a, fa = m, fm
    raise RuntimeError('Nuk u arrit toleranca.')

f = lambda x: x*x-2.0
rrenja, hist_b = pergjysmimi(f, 1.0, 2.0, tol=1e-10)
print(rrenja, 'hapa =', len(hist_b), 'mbetja =', abs(f(rrenja)))


In [ ]:
for k, a, b, m, fm in hist_b[:6]:
    print(f'{k:2d}: [{a:.6f}, {b:.6f}], m={m:.6f}, f(m)={fm:+.3e}')


## 3. Metoda e Njutonit dhe konvergjenca lokale

Linearizojmë $f$ rreth $x_n$:

$$f(x)\approx f(x_n)+f'(x_n)(x-x_n).$$

Duke kërkuar zeron e kësaj vije:

$$x_{n+1}=x_n-\frac{f(x_n)}{f'(x_n)}.$$

Nëse rrënja është e thjeshtë, $f$ është mjaft e lëmuar dhe $x_0$ është afër saj, gabimi plotëson afërsisht

$$|e_{n+1}|\approx C|e_n|^2.$$

Kjo është konvergjencë kuadratike: numri i shifrave të sakta afërsisht dyfishohet. Garancia është lokale; një pikë fillestare e dobët ose një derivat i vogël mund ta destabilizojë iteracionin.


In [ ]:
def njutoni(f, df, x0, tol=1e-12, max_iter=30):
    x = float(x0)
    historia = [x]
    for _ in range(max_iter):
        dfx = df(x)
        if abs(dfx) < 1e-14:
            raise ZeroDivisionError('Derivati është tepër i vogël.')
        x_ri = x-f(x)/dfx
        historia.append(x_ri)
        if abs(x_ri-x) <= tol*(1+abs(x_ri)):
            return x_ri, np.array(historia)
        x = x_ri
    raise RuntimeError('Nuk u arrit konvergjenca.')

rr_n, hist_n = njutoni(f, lambda x: 2*x, x0=1.0)
for k, xk in enumerate(hist_n):
    print(f'{k:2d}: x={xk:.15f}, gabimi={abs(xk-np.sqrt(2)):.3e}')


In [ ]:
# Rendi i vëzhguar q vlerësohet nga tri gabime të njëpasnjëshme.
e = np.abs(hist_n-np.sqrt(2))
for k in range(2, len(e)-1):
    if e[k-1] > 0 and e[k] > 0 and e[k+1] > 0:
        q = np.log(e[k+1]/e[k])/np.log(e[k]/e[k-1])
        print(f'k={k}: q≈{q:.3f}')


## Si do ta llogarisim kompleksitetin

Kompleksiteti nuk përcaktohet fillimisht me kronometër. Së pari zgjedhim madhësinë e hyrjes $n$, identifikojmë veprimin dominues dhe nxjerrim formulën $T(n)$. Vetëm pasi kemi një parashikim matematikor kryejmë kohëmatjen.

Për një cikël që kryen $c$ veprime në secilin prej $n$ hapave,

$$T(n)=cn+d\in\Theta(n).$$

Për dy cikle të plota të ndërfutura,

$$T(n)=\sum_{i=1}^{n}\sum_{j=1}^{n}c=cn^2\in\Theta(n^2).$$

Koha e matur modelohet si $t(n)\approx Cn^p$. Në koordinata log–log,

$$\log t=\log C+p\log n,$$

prandaj pjerrësia $p$ krahasohet me rendin e parashikuar. Matja mund të ndikohet nga kostot fikse, interpretuesi dhe memoria; ajo është kontroll empirik, jo zëvendësim i derivimit.


### 4.1 Kompleksiteti i dy metodave të rrënjës

Përgjysmimi kërkon

$$k=\Theta\left(\log\frac{b-a}{\varepsilon}\right)$$

vlerësime të funksionit. Njutoni ka zakonisht pak iteracione, por çdo hap kërkon $f$ dhe $f'$. Nëse funksioni është skalar dhe i lirë, kostoja për hap është konstante. Në sistemet jolineare, zgjidhja e sistemit të Jakobit mund të dominojë me $\Theta(n^3)$.

Krahasimi i drejtë duhet të numërojë jo vetëm iteracionet, por edhe koston e një iteracioni.


In [ ]:
tolerancat = 10.0**(-np.arange(2, 13))
hapa_b = []
hapa_n = []
for tol in tolerancat:
    _, hb = pergjysmimi(f, 1, 2, tol=tol)
    _, hn = njutoni(f, lambda x: 2*x, 1.0, tol=tol)
    hapa_b.append(len(hb)); hapa_n.append(len(hn)-1)

fig, ax = plt.subplots(figsize=(7,4))
ax.semilogx(tolerancat, hapa_b, 'o-', label='përgjysmimi')
ax.semilogx(tolerancat, hapa_n, 's-', label='Njutoni')
ax.invert_xaxis(); ax.set(xlabel='toleranca', ylabel='numri i hapave')
ax.grid(alpha=0.3); ax.legend(); plt.show()


## 5. Sistemet lineare: eliminimi i Gausit me dorë

Në kolonën e parë eliminohen $n-1$ elemente; në të dytën $n-2$, e kështu me radhë. Përditësimi i një rreshti prek afërsisht $n-k$ elemente. Numri dominues i veprimeve është

$$\sum_{k=1}^{n-1}(n-k)^2
=\sum_{j=1}^{n-1}j^2
=\frac{(n-1)n(2n-1)}{6}
\sim\frac{n^3}{3}.$$

Duke përfshirë shumëzimet dhe zbritjet, faktori klasik është rreth $2n^3/3$. Pra faktorizimi është $\Theta(n^3)$; zëvendësimi prapa është $\Theta(n^2)$.


In [ ]:
def gauss_pivot(A, b):
    A = np.array(A, dtype=float, copy=True)
    b = np.array(b, dtype=float, copy=True)
    n = len(b)
    for k in range(n-1):
        p = k + np.argmax(np.abs(A[k:, k]))  # pivotimi i pjesshëm
        if abs(A[p, k]) < 1e-15:
            raise np.linalg.LinAlgError('Matrica është singulare.')
        A[[k,p]], b[[k,p]] = A[[p,k]], b[[p,k]]
        for i in range(k+1, n):
            m = A[i,k]/A[k,k]
            A[i,k:] -= m*A[k,k:]
            b[i] -= m*b[k]
    x = np.empty(n)
    for i in range(n-1, -1, -1):
        x[i] = (b[i]-A[i,i+1:]@x[i+1:])/A[i,i]
    return x

A = np.array([[1e-16, 1.0], [1.0, 1.0]])
b = np.array([1.0, 2.0])
x = gauss_pivot(A, b)
print('x =', x, 'mbetja =', np.linalg.norm(A@x-b))


## Si do ta analizojmë qëndrueshmërinë

**Kushtëzimi** i përket problemit: sa ndryshon zgjidhja e saktë kur të dhënat perturbohen pak? **Qëndrueshmëria** i përket algoritmit: sa gabim shton procedura e llogaritjes? Në trajtë skematike,

$$\text{gabimi përpara}\lesssim
\text{numri i kushtëzimit}\times\text{gabimi prapa}.$$

Në secilin laborator do të përdorim një provë konkrete: mbetje, ligj ruajtjeje, kufi stabiliteti, rezultat analitik ose përsëritje me hap më të vogël. Fjalia “grafiku duket mirë” nuk është provë numerike.


### 6.1 Mbetja dhe numri i kushtëzimit

Për $A\mathbf{x}=\mathbf{b}$, perturbimet relative lidhen afërsisht me

$$\frac{\|\delta\mathbf{x}\|}{\|\mathbf{x}\|}
\lesssim\kappa(A)\frac{\|\delta\mathbf{b}\|}{\|\mathbf{b}\|},$$

ku $\kappa(A)=\|A\|\|A^{-1}\|$. Një mbetje e vogël $\mathbf{r}=\mathbf{b}-A\widehat{\mathbf{x}}$ mund të bashkëjetojë me gabim të madh kur $\kappa(A)$ është e madhe. Pivotimi përmirëson qëndrueshmërinë e eliminimit; ai nuk e ndryshon kushtëzimin e problemit.


In [ ]:
for delta in [1e-2, 1e-6, 1e-10, 1e-14]:
    A = np.array([[1.0, 1.0], [1.0, 1.0+delta]])
    x_sakte = np.array([1.0, 1.0])
    b = A@x_sakte
    b_pert = b + np.array([0.0, 1e-12])
    x_hat = np.linalg.solve(A, b_pert)
    print(f'delta={delta:.0e}, kappa={np.linalg.cond(A):.3e}, '
          f'gabimi={np.linalg.norm(x_hat-x_sakte):.3e}, '
          f'mbetja={np.linalg.norm(A@x_hat-b_pert):.3e}')


## 7. Vlera vetjake dominuese: metoda fuqi

Nëse $A$ ka vlera vetjake $|\lambda_1|>|\lambda_2|\geq\cdots$ dhe vektori fillestar ka përbërës në drejtimin e $\mathbf{v}_1$, atëherë

$$A^k\mathbf{x}_0
=c_1\lambda_1^k\left[\mathbf{v}_1+
\sum_{j>1}\frac{c_j}{c_1}\left(\frac{\lambda_j}{\lambda_1}\right)^k\mathbf{v}_j\right].$$

Normalizimi pengon tejmbushjen. Gabimi ulet afërsisht me $|\lambda_2/\lambda_1|^k$. Për matricë të dendur, produkti matricë–vektor kushton $\Theta(n^2)$ për iteracion; për $k$ iteracione, $\Theta(kn^2)$.


In [ ]:
def metoda_fuqi(A, x0=None, tol=1e-10, max_iter=1000):
    A = np.asarray(A, float); n = A.shape[0]
    x = np.ones(n) if x0 is None else np.asarray(x0, float)
    x /= np.linalg.norm(x)
    historia = []
    for k in range(max_iter):
        y = A@x
        x = y/np.linalg.norm(y)          # normalizim në çdo hap
        lam = x@(A@x)                    # herësi i Rayleigh-it
        r = np.linalg.norm(A@x-lam*x)
        historia.append(r)
        if r <= tol:
            return lam, x, np.array(historia)
    return lam, x, np.array(historia)

A = np.array([[2.0,1.0],[1.0,3.0]])
lam, vec, hist = metoda_fuqi(A)
print('lambda =', lam, 'vektori =', vec, 'iteracione =', len(hist))


In [ ]:
fig, ax = plt.subplots(figsize=(6.5,4))
ax.semilogy(np.arange(1,len(hist)+1), hist, 'o-')
ax.set(xlabel='iteracioni', ylabel='mbetja vetjake', title='Konvergjenca e metodës fuqi')
ax.grid(alpha=0.3); plt.show()


## 8. Shembulli i plotë dhe punishte

Shembulli i plotë lidh tri idetë: rrënja përdoret për një ekuacion ekuilibri, sistemi linear për korrigjimin lokal dhe metoda fuqi për shkallën dominuese. Gjatë orës ndryshoni vetëm një nga parametrat:

- pikën fillestare të Njutonit;
- tolerancën e përgjysmimit;
- perturbimin e matricës;
- raportin $|\lambda_2/\lambda_1|$.

Parashikoni para ekzekutimit se cili ndryshim prek numrin e iteracioneve dhe cili prek saktësinë.


In [ ]:
def krahaso_rrenjet(x0_lista=(-2.0, 0.2, 1.0, 4.0)):
    f = lambda x: x**3-x-1
    df = lambda x: 3*x*x-1
    rezultate = []
    for x0 in x0_lista:
        try:
            r, h = njutoni(f, df, x0, max_iter=50)
            rezultate.append((x0, r, len(h)-1, abs(f(r))))
        except Exception as e:
            rezultate.append((x0, np.nan, 0, str(e)))
    return rezultate

for row in krahaso_rrenjet(): print(row)


### Kriteret e ndalimit nuk janë të barasvlershme

Kushti $|x_{n+1}-x_n|<\varepsilon$ mat ndryshimin e iteracionit, ndërsa $|f(x_n)|<\varepsilon_f$ mat mbetjen. Në një rrënjë ku $|f'(x_\star)|$ është shumë i vogël, një mbetje e vogël mund të korrespondojë me gabim më të madh në $x$. Linearizimi jep

$$|x-x_\star|\approx\frac{|f(x)|}{|f'(x_\star)|}.$$

Prandaj një zbatim serioz kontrollon si hapin, ashtu edhe mbetjen dhe numrin maksimal të iteracioneve.


In [ ]:
# Krahasojmë kriterin e hapit me mbetjen për një funksion të sheshtë pranë rrënjës.
f_flat=lambda x:(x-1.0)**3
df_flat=lambda x:3*(x-1.0)**2
r,h=njutoni(f_flat,df_flat,1.5,tol=1e-6,max_iter=100)
print('rrënja e llogaritur=',r,'gabimi=',abs(r-1),'mbetja=',abs(f_flat(r)),'hapa=',len(h)-1)


### Kostoja reale e një sistemi linear

Faktorizimi $PA=LU$ kushton $O(n^3)$, por pasi ai është kryer, çdo anë e djathtë e re zgjidhet me dy zëvendësime me kosto $O(n^2)$. Në probleme ku e njëjta matricë përdoret për shumë vektorë $b$, faktorizimi duhet ripërdorur. Llogaritja e anasjelltës së plotë është më e shtrenjtë, përdor më shumë kujtesë dhe zakonisht është më pak e qëndrueshme se zgjidhja e drejtpërdrejtë.


In [ ]:
# Verifikojmë lidhjen midis kushtëzimit dhe amplifikimit të perturbimit.
for kappa_target in [1e2,1e6,1e10]:
    A=np.diag([1.0,1.0/kappa_target]); b=np.ones(2); x=np.linalg.solve(A,b)
    db=np.array([0.0,1e-12]); xp=np.linalg.solve(A,b+db)
    rel_b=np.linalg.norm(db)/np.linalg.norm(b)
    rel_x=np.linalg.norm(xp-x)/np.linalg.norm(x)
    print(f'kappa={np.linalg.cond(A):.1e}, amplifikimi={rel_x/rel_b:.2e}')


### Si raportohet një zgjidhje iterative

Raportoni vlerën e përafruar, mbetjen, numrin e iteracioneve, tolerancën dhe arsyen e ndalimit. Për metodën fuqi raportoni edhe raportin spektral kur mund të vlerësohet. Dy matrica të së njëjtës madhësi mund të kërkojnë numër shumë të ndryshëm iteracionesh, sepse konvergjenca kontrollohet nga $|\lambda_2/\lambda_1|$, jo vetëm nga $n$.


In [ ]:
for ratio in [.2,.8,.98]:
    A=np.diag([1.0,ratio]); lam,v,h=metoda_fuqi(A,x0=[1,1],tol=1e-8,max_iter=10000)
    print(f'|lambda2/lambda1|={ratio:.2f}, iteracione={len(h)}, mbetja={h[-1]:.2e}')


### Pyetje për vetëkontroll

1. Pse përgjysmimi është më i ngadaltë, por më i parashikueshëm se Njutoni?
2. Cila pjesë e eliminimit jep termin dominues $n^3$?
3. Pse pivotimi nuk e “shëron” një problem të keqkushtëzuar?
4. Çfarë ndodh me metodën fuqi kur dy vlerat vetjake kryesore kanë module pothuajse të barabarta?


# Pjesa II — GNU Octave

## 9. Përgjysmimi dhe Njutoni

```octave
f=@(x) x.^2-2; df=@(x) 2*x;
a=1; b=2; tol=1e-10;
while (b-a)/2>tol
    m=(a+b)/2;
    if f(a)*f(m)<0, b=m; else, a=m; end
end
fprintf('pergjysmimi: %.15f\n',(a+b)/2);

x=1.0;
for k=1:20
    xri=x-f(x)/df(x);
    if abs(xri-x)<=tol*(1+abs(xri)), break; end
    x=xri;
end
fprintf('Njutoni: %.15f, hapa=%d\n',xri,k);
```

## 10. Sistemi linear dhe vlera vetjake

```octave
A=[1,1;1,1+1e-10]; b=A*[1;1];
x=A\b;
fprintf('kappa = %.3e, mbetja = %.3e\n',cond(A),norm(A*x-b));

A=[2,1;1,3]; x=[1;1]; x=x/norm(x);
for k=1:100
    x=A*x; x=x/norm(x);
    lambda=x'*(A*x);
    r=norm(A*x-lambda*x);
    if r<1e-10, break; end
end
fprintf('lambda = %.12g, mbetja = %.3e\n',lambda,r);
```


## Ekzekutimi jashtë notebook-ut

Një notebook e ndihmon shpjegimin, por studenti duhet të dijë të ekzekutojë edhe një program të zakonshëm. Qeliza vijuese krijon një skript minimal në `scripts_local/`. Ai përmban eksperimentin qendror dhe mund të ndryshohet me një editor teksti.


In [ ]:
script_dir = Path('scripts_local')
script_dir.mkdir(exist_ok=True)
script_text = "import numpy as np\n\ndef bisect(f,a,b,tol):\n    while (b-a)/2>tol:\n        m=(a+b)/2\n        if f(a)*f(m)<0: b=m\n        else: a=m\n    return (a+b)/2\n\nf=lambda x:x*x-2\nfor tol in [1e-3,1e-6,1e-9]: print(tol,bisect(f,1,2,tol))\nA=np.array([[2.,1.],[1.,3.]])\nprint('eig=',np.linalg.eigvalsh(A))\n"
script_path = script_dir / 'laboratori_03_ekuacionet_sistemet_vlerat_vetjake.py'
script_path.write_text(script_text, encoding='utf-8')
print('U krijua:', script_path)


Ekzekutimi në terminal:

```bash
python3 scripts_local/laboratori_03_ekuacionet_sistemet_vlerat_vetjake.py
```

Mos raportoni vetëm se programi u ekzekutua. Shënoni komandën, një dalje numerike, një kontroll dhe një përfundim.

## Raporti i shkurtër dhe kontrolli përfundimtar

Raporti duhet të përmbajë pyetjen, modelin, supozimet, derivimin kryesor, analizën e kompleksitetit, provën e qëndrueshmërisë, figurën me boshte e njësi dhe interpretimin fizik.

- [ ] Nxora me dorë numrin e hapave të përgjysmimit.
- [ ] Mata mbetjen dhe nuk u mjaftova me numrin e iteracioneve.
- [ ] Nxora koston $\Theta(n^3)$ të eliminimit.
- [ ] Dallova pivotimin nga kushtëzimi i matricës.
- [ ] Ekzekutova skriptin Python nga terminali.
- [ ] Identifikova fragmentin përkatës në GNU Octave.
